# Silver — ecommerce_itens_pedido

Este notebook lê a Bronze Delta `squad1.bronze_ecommerce_itens_pedido`, aplica as 10 regras de qualidade da tabela de itens de pedido, grava a Silver Delta e registra os resultados na tabela compartilhada `squad1.dq_monitoring_logs`.




In [0]:
%run ../utils/utils

## Imports e parâmetros

In [0]:

import uuid
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime, timezone

# Variáveis
RUN_ID = str(uuid.uuid4())
TABELA_ALVO = "ecommerce_itens_pedido"
TABELA_DQ = "dq_monitoring_logs"

## Setup e Referências

In [0]:
# 1. Carrega a tabela Bronze de Itens de Pedido
try:
    df_bronze_itens = ler_delta("bronze", TABELA_ALVO, STORAGE_OPTIONS)
except Exception as e:
    raise Exception(f"Erro: A tabela Bronze de {TABELA_ALVO} não foi encontrada.")

# 2. Isola o Micro-lote (Considerando Silver E Quarentena)
if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_silver_atual = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    
    if delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):
        df_quarentena = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS)
        df_processados = df_silver_atual.select("id_item_pedido") \
            .union(df_quarentena.select("id_item_pedido"))
    else:
        df_processados = df_silver_atual.select("id_item_pedido")
        
    df_micro_lote = df_bronze_itens.join(df_processados, "id_item_pedido", "left_anti")
else:
    df_micro_lote = df_bronze_itens

qtd_novos = df_micro_lote.count()
print(f"Registros novos no micro-lote para processar: {qtd_novos}")

# =================================================================================
# 3. Leitura das Tabelas de Referência Cruzada (FKS)
# =================================================================================
# CORREÇÃO R2/R3: ambas agora priorizam a Silver das tabelas referenciadas,
# caindo para a Bronze quando a Silver ainda não existir.
df_pedidos_ref = obter_referencia_silver_ou_bronze("ecommerce_pedidos", ["id_pedido", "valor_total"]) \
    .withColumnRenamed("valor_total", "valor_total_pedido")
 
df_produtos_ref = obter_referencia_silver_ou_bronze("ecommerce_produtos", ["sku"]) \
    .withColumn("sku_existe", F.lit(True))

print("Tabelas de referência para validação cruzada carregadas com sucesso.")

## A Muralha de Qualidade (Regras de 1 a 10)

In [0]:

logs_r10 = []

if delta_existe("bronze", "ecommerce_itens_pedido", STORAGE_OPTIONS):
    df_todos_itens = ler_delta("bronze", "ecommerce_itens_pedido", STORAGE_OPTIONS) \
        .select("id_pedido").dropDuplicates()
else:
    schema_vazio = StructType([StructField("id_pedido", LongType(), True)])
    df_todos_itens = spark.createDataFrame([], schema_vazio)

df_pedidos_sem_item = df_pedidos_ref.join(df_todos_itens, "id_pedido", "left_anti")
qtd_pedidos_sem_item = df_pedidos_sem_item.count()
total_pedidos = df_pedidos_ref.count()

if qtd_pedidos_sem_item > 0:
    logs_r10.append((
        RUN_ID, TABELA_ALVO, "R10_PEDIDO_SEM_ITENS", "FAIL", "Critica",
        int(qtd_pedidos_sem_item), int(total_pedidos), datetime.now(timezone.utc),
        "Bronze Delta (ecommerce_pedidos vs ecommerce_itens_pedido)"
    ))
    print(f"Atenção: {qtd_pedidos_sem_item} pedido(s) sem nenhum item associado (R10).")
else:
    print("R10: nenhum pedido sem item encontrado.")


# ==============================================================================
# REGRAS 1-9 (nível item) — só rodam quando há itens novos no micro-lote.
# Ao final, o log da R10 (calculado acima, incondicionalmente) é sempre
# mesclado ao resultado, independente de ter havido itens novos ou não.
# ==============================================================================
if qtd_novos > 0:
    from functools import reduce

    # CORREÇÃO R1: window agora tem orderBy explícito por bronze_ingested_at,
    # para que "a primeira ocorrência" seja determinística (a mesma linha
    # sempre "ganha" entre execuções, em vez de depender de ordem arbitrária
    # de leitura do Spark). row_number() == 1 marca o sobrevivente; qualquer
    # linha com row_number() > 1 é duplicata e é reprovada.
    w_pedido = Window.partitionBy("id_pedido")
    w_item = Window.partitionBy("id_item_pedido").orderBy(F.col("bronze_ingested_at").asc())

    # Prepara o DataFrame base aplicando as transformações de tipo e Joins
    df_base = df_micro_lote \
        .join(df_pedidos_ref, "id_pedido", "left") \
        .join(df_produtos_ref, "sku", "left") \
        .withColumn("valor_item_liquido", (F.col("preco_unitario") - F.coalesce(F.col("desconto_aplicado"), F.lit(0.0))) * F.col("quantidade")) \
        .withColumn("soma_valor_itens", F.sum("valor_item_liquido").over(w_pedido)) \
        .withColumn("qtd_itens_por_pedido", F.count("id_item_pedido").over(w_pedido)) \
        .withColumn("row_item_no_grupo", F.row_number().over(w_item))

    # Aplicação das regras de qualidade de Itens de Pedido (nível item)
    df_silver_itens = df_base \
        .withColumn(
            "r1_id_item_pedido_falhou",
            # Nulo sempre reprova; duplicata só reprova a partir da SEGUNDA
            # ocorrência (row_item_no_grupo > 1). A primeira ocorrência
            # (row_item_no_grupo == 1) é validada.
            F.col("id_item_pedido").isNull() | (F.col("row_item_no_grupo") > 1)
        ) \
        .withColumn("r2_id_pedido_fk_falhou", F.col("id_pedido").isNull() | F.col("valor_total_pedido").isNull()) \
        .withColumn("r3_sku_fk_falhou", F.col("sku").isNull() | F.col("sku_existe").isNull()) \
        .withColumn(
            "r4_quantidade_falhou",
            F.col("quantidade").isNull() |
            (F.col("quantidade") < 1) |
            (F.col("quantidade") != F.floor(F.col("quantidade")))
        ) \
        .withColumn("r5_preco_unitario_falhou", F.col("preco_unitario").isNull() | (F.col("preco_unitario") <= 0)) \
        .withColumn("r6_desconto_teto_falhou", F.col("desconto_aplicado").isNotNull() & (F.col("desconto_aplicado") > F.col("preco_unitario"))) \
        .withColumn(
            "r7_consistencia_financeira_falhou",
            F.col("valor_total_pedido").isNotNull() & (F.abs(F.col("soma_valor_itens") - F.col("valor_total_pedido")) > 0.01)
        ) \
        .withColumn("r8_desconto_negativo_falhou", F.col("desconto_aplicado").isNotNull() & (F.col("desconto_aplicado") < 0)) \
        .withColumn(
            "r9_desconto_limite_50_falhou",
            F.col("desconto_aplicado").isNotNull() &
            (F.coalesce(F.try_divide(F.col("desconto_aplicado"), F.col("preco_unitario")), F.lit(1.0)) > 0.5)
        )

    catalogo_regras = [
        {"coluna": "r1_id_item_pedido_falhou", "regra": "R1_ID_ITEM_NULO_DUPLICADO", "severidade": "Critica"},
        {"coluna": "r2_id_pedido_fk_falhou", "regra": "R2_PEDIDO_FK_ORFAO", "severidade": "Critica"},
        {"coluna": "r3_sku_fk_falhou", "regra": "R3_SKU_FK_ORFAO", "severidade": "Critica"},
        {"coluna": "r4_quantidade_falhou", "regra": "R4_QUANTIDADE_INVALIDA", "severidade": "Critica"},
        {"coluna": "r5_preco_unitario_falhou", "regra": "R5_PRECO_UNITARIO_NEGATIVO_ZULO", "severidade": "Critica"},
        {"coluna": "r6_desconto_teto_falhou", "regra": "R6_DESCONTO_MAIOR_QUE_PRECO", "severidade": "Critica"},
        {"coluna": "r7_consistencia_financeira_falhou", "regra": "R7_DIVERGENCIA_TOTAL_PEDIDO", "severidade": "Critica"},
        {"coluna": "r8_desconto_negativo_falhou", "regra": "R8_DESCONTO_NEGATIVO", "severidade": "Critica"},
        {"coluna": "r9_desconto_limite_50_falhou", "regra": "R9_DESCONTO_SUPERIOR_A_50_PORCENTO", "severidade": "Critica"},
    ]

    total_registros = df_silver_itens.count()
    logs_list = []

    for r in catalogo_regras:
        qtd_falhas = df_silver_itens.filter(F.col(r["coluna"]) == True).count()
        if qtd_falhas > 0:
            logs_list.append((
                RUN_ID, TABELA_ALVO, r["regra"], "FAIL", r["severidade"],
                int(qtd_falhas), int(total_registros), datetime.now(timezone.utc), f"Bronze Delta ({TABELA_ALVO})"
            ))

    # Mescla os logs das regras 1-9 com o log da R10
    logs_list = logs_list + logs_r10

    if logs_list:
        df_dq_monitoring_logs_novos = spark.createDataFrame(logs_list, schema_dq_logs())
    else:
        df_dq_monitoring_logs_novos = spark.createDataFrame([], schema_dq_logs())

    # --- UNIFICAÇÃO TOTAL DAS FALHAS (Nível Item) ---
    todas_flags = [r["coluna"] for r in catalogo_regras]
    condicao_total_falha = reduce(lambda a, b: a | b, [F.col(c) for c in todas_flags])

    df_silver_itens = (df_silver_itens
        .withColumn("silver_linha_valida", ~condicao_total_falha)
        .withColumn("silver_processed_at", F.current_timestamp())
        .withColumn("silver_run_id", F.lit(RUN_ID)))

    print("Muralha de qualidade unificada para Itens de Pedido estruturada. "
          "R1: primeira ocorrência de id_item_pedido duplicado é validada, as seguintes são reprovadas.")
else:
    if logs_r10:
        df_dq_monitoring_logs_novos = spark.createDataFrame(logs_r10, schema_dq_logs())
        print("Nenhum item novo, mas R10 encontrou pedido(s) sem item — log será gravado.")
    else:
        df_dq_monitoring_logs_novos = spark.createDataFrame([], schema_dq_logs())
    print("Nenhum registro novo encontrado para processamento (regras 1-9 ignoradas nesta execução).")

## Gravação Blindada e Logs

In [0]:
if qtd_novos > 0:
    # CORREÇÃO: "silver_tem_aviso" removido da lista para adequação ao novo contrato
    colunas_finais = df_micro_lote.columns + ["silver_processed_at", "silver_run_id"]

    # ---------------- 1. GRAVAÇÃO DOS VÁLIDOS ---------------- #
    df_silver_validos = (df_silver_itens
        .filter(F.col("silver_linha_valida") == True)
        .select(*colunas_finais))

    qtd_validos = df_silver_validos.count()
    print(f"Registros aprovados para a Silver: {qtd_validos}")

    if qtd_validos > 0:
        sucesso_silver = gravar_delta(
            df=df_silver_validos, camada="silver", tabela=TABELA_ALVO,
            storage_opts=STORAGE_OPTIONS, mode="append", particionar=True
        )
        if sucesso_silver:
            print(f"Tabela Silver {TABELA_ALVO} atualizada com sucesso!")

    # ---------------- 2. GRAVAÇÃO DA QUARENTENA (DEDUPLICADA) ---------------- #
    df_silver_invalidos = (df_silver_itens
        .filter(F.col("silver_linha_valida") == False)
        .select(*colunas_finais))

    if df_silver_invalidos.count() > 0:
        # CORREÇÃO: indentação alinhada com o "else" logo abaixo (estava com
        # 1 espaço a menos, causando IndentationError).
        if delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):
            df_quarentena_historico = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS)

            df_quarentena_para_gravar = df_silver_invalidos.join(
                df_quarentena_historico.select("id_item_pedido"),
                on="id_item_pedido",
                how="left_anti"
            )
        else:
            df_quarentena_para_gravar = df_silver_invalidos

        qtd_novos_rejeitados = df_quarentena_para_gravar.count()
        if qtd_novos_rejeitados > 0:
            sucesso_quarentena = gravar_delta(
                df=df_quarentena_para_gravar,
                camada="silver/quarentena",
                tabela=TABELA_ALVO,
                storage_opts=STORAGE_OPTIONS,
                mode="append",
                particionar=False
            )
            if sucesso_quarentena:
                print(f"Enviados {qtd_novos_rejeitados} registros novos para a quarentena.")
        else:
            print("Todos os registros reprovados já existiam na quarentena histórica.")

    # ---------------- 3. GRAVAÇÃO DOS LOGS NA RAIZ ---------------- #
    if 'df_dq_monitoring_logs_novos' in locals() and df_dq_monitoring_logs_novos.count() > 0:
        sucesso_logs = gravar_delta(
            df=df_dq_monitoring_logs_novos,
            camada="",
            tabela=TABELA_DQ,
            storage_opts=STORAGE_OPTIONS,
            mode="append",
            particionar=False
        )
        if sucesso_logs:
            print("Logs de qualidade consolidados na raiz!")
else:
    print("Rotina finalizada sem alterações físicas.")

## Validação final

In [0]:
print("===== VALIDAÇÃO FINAL =====")

if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_validacao_silver = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    print(f"Registros totais na Silver {TABELA_ALVO}:", df_validacao_silver.count())
    display(df_validacao_silver.limit(20))
else:
    print(f"Aviso: tabela Silver {TABELA_ALVO} não encontrada.")


if delta_existe("", "dq_monitoring_logs", STORAGE_OPTIONS):
    df_logs_validacao = ler_delta("", "dq_monitoring_logs", STORAGE_OPTIONS) \
        .filter(F.col("tabela") == TABELA_ALVO)
    print(f"Total de violações registradas para {TABELA_ALVO}:", df_logs_validacao.count())
    display(df_logs_validacao.orderBy(F.col("timestamp_execucao").desc()).limit(20))
else:
    print("Aviso: tabela dq_monitoring_logs não encontrada na raiz do Data Lake.")